<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title"><b>Modeling with Gurobi: the basics</b></span> by <a xmlns:cc="http://creativecommons.org/ns#" href="http://mate.unipv.it/gualandi" property="cc:attributionName" rel="cc:attributionURL">Stefano Gualandi</a> is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>.<br />Based on a work at <a xmlns:dct="http://purl.org/dc/terms/" href="https://github.com/mathcoding/opt4ds" rel="dct:source">https://github.com/mathcoding/opt4ds</a>.

**NOTE:** Execute the following command when running this notebook in Google Colab.

In [ ]:
# Run if on Colab
# %pip install gurobipy

# 2. Writing LP models with Gurobi
In this notebook, we show how to use Gurobi to write general LP problems.

## 2.1 Steel Production Planning

In this section, we explain how to solve the **Linear Programming** model we wrote for the Steel Planning problem in class (see the slides on KIRO). This problem is given in Exercise 1.1 in Chapter 1 of [Linear Programming, Foundations and Extensions](https://link.springer.com/book/10.1007/978-1-4614-7630-6) by [R.J. Vanderbei](https://vanderbei.princeton.edu/).

We show below how to use [Gurobi](https://www.gurobi.com/academia/academic-program-and-licenses/) to define the **variables**, the **objective function**, and the **constraints**.

### 2.1.1 Software Installation
First, we need to install [Gurobi](http://www.gurobi.org/). If you are running this notebook in Colab, you don't need to install anything else on your computer.

The following line installs the free version of Gurobi in Google Colab or on your computer.

In [ ]:
# Run if on Colab
%pip install gurobipy

To install an academic license of Gurobi on your computer, follow these [instructions](https://www.gurobi.com/features/academic-named-user-license/).

### 2.1.2 Define the data
Recall that a possible model of the steel planning problem is as follows:

$$
\begin{align}
\max \quad & p_B x_B + p_C x_C  \\
\quad & \frac{x_B}{r_B} + \frac{x_C}{r_C} \leq T \\
& 0 \leq x_B \leq d_B\\
& 0 \leq x_C \leq d_C
\end{align}
$$

First, we need to define the data for our instance:

In [ ]:
# Data
pB = 25     # Profit of band ($ profit per ton)
pC = 30     # Profit of coil ($ profit per ton)

rB = 200    # Production rate of band (tons per hour)
rC = 140    # Production rate of coil (tons per hour)

dB = 6000   # Maximum demand for band (in tons)
dC = 4000   # Maximum demand for coil (in tons)

T = 40      # Total hours available (per week)

### 2.1.3 Define the model entities
To build the Linear Programming model with Gurobi, we first need to import the Gurobi library:

In [ ]:
from gurobipy import Model, GRB

At this point, we declare a global object that refers to our model by creating an instance of the class [Model](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model):

In [ ]:
model = Model()

Notice that `Model` is a Python class, and we are initializing an object called `model` of type `Model`.

Then, we declare the two nonnegative variables, along with their cost coefficients and type, using the [model.addVar](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.addVar) method:

In [ ]:
# Declare the decision variables
xB = model.addVar(lb=0, ub=dB, obj=pB, vtype=GRB.CONTINUOUS, name='xB')
xC = model.addVar(lb=0, ub=dC, obj=pC, vtype=GRB.CONTINUOUS, name='xC')

Here, we add variables `xB` and `xC` to the model. The two variables are of type `GRB.CONTINUOUS`, and we are declaring the two variables $0 \leq x_B \leq U_B$ and $0 \leq x_C \leq U_C$. The cost coefficients are `pB` and `pC`.

We need to specify the direction of the objective function: `min` or `max`.

In [ ]:
model.ModelSense = GRB.MAXIMIZE

The next step is to introduce the linear constraint using the [model.addConstr](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.addConstr) method:

In [ ]:
# Declare constraints
model.addConstr(1/rB * xB + 1/rC * xC <= T)

In [ ]:
# To finalize the model, call:
model.update()

Notice that we are **declaring** the model, without programming any algorithm to actually solve it. To find the optimal solution of this LP, we call `model.optimize()`.

### 2.1.4 Solve the model
We have used the Gurobi Python library to *declare* our Linear Programming model. Next, we use the Gurobi solver to actually find the optimal values for the decision variables.

In [ ]:
# Solver call
model.optimize()

Every time we invoke a solver, it is very good practice to check the status of the solver, since it may have stopped its execution for several different reasons:

In [ ]:
# Basic info about the solution process
print(f"Status: {model.Status}, ObjVal: {model.ObjVal}")

Be aware of the meaning of the different status codes by checking the [Optimization Status Code](https://docs.gurobi.com/projects/optimizer/en/current/reference/numericcodes/statuscodes.html#optimization-status-codes).

Whenever the status of the solver is `2=OPTIMAL`, you can query the solver to get the values of the decision variables.

In [ ]:
# Report solution value
print("Decision variables:")
print("\tProduction of bands:", xB.X)
print("\tProduction of coils:", xC.X)

**REMARK:** To check the problem actually solved by Gurobi in the standard LP format, you can write the following command:

In [ ]:
model.write('production.lp')

To check where the file was written:

In [ ]:
!ls

### 2.1.5 Complete Script
We report the complete script below.

In [ ]:
# Import the library
from gurobipy import Model, GRB

# Create an instance of the model object
model = Model()

# Declare the decision variables
xB = model.addVar(lb=0, ub=dB, obj=pB, vtype=GRB.CONTINUOUS, name='xB')
xC = model.addVar(lb=0, ub=dC, obj=pC, vtype=GRB.CONTINUOUS, name='xC')

# Specify the objective function direction
model.ModelSense = GRB.MAXIMIZE

# Declare the single linear constraint
model.addConstr(1/rB * xB + 1/rC * xC <= T)

# Solver call
model.optimize()

# Basic info about the solution process
print(f"Status: {model.Status}, ObjVal: {model.ObjVal}")

# Report value of the decision variables
print("Decision variables:")
print("\tProduction of bands:", xB.X)
print("\tProduction of coils:", xC.X)

## 2.2 Lego Planning Problem
As a first exercise, you have to solve the **Linear Programming (LP)** problem that we have written to model the Lego Planning problem (see the slides on KIRO):

$$
\begin{align}
\max \quad & 8 c + 11 t  \\
 \quad & 2c + 2t \leq 24 \\
& c + 2 t \leq 18\\
& c \geq 0\\
& t \geq 0
\end{align}
$$

You have to use Gurobi to define the **variables**, the **objective function**, and the **constraints**.

**EXERCISE 1:** Using the template for the *Steel Production Planning Problem*, solve the LEGO instance defined above.

In [ ]:
# Write your own script

**EXERCISE 2:** Modify the previous script to solve the second version of the Lego Planning problem:

$$
\begin{align}
\max \quad & 8 c + 11 t + 15 s \\
 \quad & 2c + 2t +2s \leq 24 \\
& c + 2 t +3s \leq 18\\
& c \geq 0\\
& t \geq 0\\
& s \geq 0
\end{align}
$$

You have to add a third variable to the model, modify the objective function and the two constraints, then call the solver again, check the solver status, and inspect the solution values.

In [ ]:
# Write your own script

## 2.3 Random LPs
Let's consider the following Python function that generates a random LP instance.

Recall that an LP is completely defined by the cost vector $c \in \mathbb{R}^n$, the RHS vector $b \in \mathbb{R}^m$, and the coefficient matrix $A \in \mathbb{R}^{m \times n}$:

$$
    z = \min \,\{ c x \mid Ax \geq b, x \geq 0 \,\}
$$

Look at the following function.

In [ ]:
from numpy import random

def RandomLP(n, m, seed=13):
    random.seed(seed)
    c = random.randint(1, 10, size=n)
    b = random.randint(1, 10, size=m)
    A = random.randint(1, 10, size=(m,n))    
    return c, b, A
    
print(RandomLP(3,4))

Next, we write another function that takes as input the data of an LP instance, builds the LP model using the Gurobi syntax, solves the instance, checks the solver status, and (if possible) prints an optimal solution.

In [ ]:
from gurobipy import Model, GRB, quicksum

def SolveLP(c, b, A):
    m = len(b)
    n = len(c)
    
    model = Model()
    
    # Be careful: Python ranges start at 0 (unlike Matlab array, which starts at 1)
    I = range(n)
    J = range(m)
    
    x = {}
    for i in I:
        x[i] = model.addVar(obj=c[i], name="x_{}".format(i))    
    
    for j in J:
        model.addConstr(quicksum(A[j, i]*x[i] for i in I) >= b[j])
    
    # Default: model.ModelSense = GRB.MINIMIZE

    model.optimize()

    # Basic info about the solution process
    print(f"Status: {model.Status}, ObjVal: {model.ObjVal}")

    xbar = [x[i].X for i in I]

    # Return objective value and decision variables
    return model.ObjVal, xbar

Below we show an example of calling this function.

In [ ]:
c, b, A = RandomLP(2, 3, 1717)
print(c,b,A)
print(SolveLP(c, b, A))

### 2.4 Nurse Scheduling
Consider the nurse scheduling problem in the lecture slides.

Let us write the same model, but in addition, we use a cost larger than 1 for the Friday, Saturday, and Sunday nights.

For the model description, we refer to the slides used during the lecture.

In [ ]:
# Import the numerical Python library https://numpy.org/
import numpy as np

def StaffScheduling():
    # Data
    d = 7
    w = 5
    demand = [5, 6, 7, 3, 3, 2, 2]
    cost = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

    # Using the numpy library, create a matrix of dimension "d x d"
    A = np.zeros( (d,d) )
    # Set the matrix coefficient to 1 using the pattern from the slides
    for i in range(d):
        for j in range(w):
            A[(i+j)%d, i] = 1
    
    # Declare the model
            
    # TODO: Complete this script using the Gurobi library

    return None, None
 

In [ ]:
print(StaffScheduling())

### 2.5 Exercise: Steel Recycle Blending Problem
Industrial steel can be easily recycled, since it is possible to melt any scrap to obtain liquid steel (without plastics, glass, ...).
However, it is hard to separate each single metal present in the scrap and, as a consequence, besides iron, we can also get chromium, nickel, and other impurities in the liquid steel.

Depending on the type of production, some metals are desirable, while others are not. For example, stainless steel 18/10 must have 18% of chromium and 10% of nickel (consider that chromium and nickel are very expensive, much more than steel itself).

**Problem Statement:** Suppose that the Brambilla Steel company of Voghera can choose to buy some scrap blocks with different properties regarding the metals contained in each block. The company wants to produce, at minimum cost, 100 quintals of stainless steel 18/10, which must have at least 65% of iron and at most 1% of impurity materials. Which fraction of each block should it buy?

The data of the problem are given below.

In [7]:
import numpy as np

# Data of the problem (in theory, read data from .csv or excel file)

# Blocks you can buy
Blocks = ['Block1','Block2','Block3','Block4','Block5','Block6']

Weights = [30, 90, 50, 70, 60, 50]  # In quintals
Costs = [50, 100, 80, 85, 92, 115]  # Thousand of euros

# Components of metal in each block (given in percentage)
Cs = np.matrix([[93, 76, 74, 65, 72, 68],  # Ferro
                [5, 13, 11, 16, 6, 23],    # Cromo
                [0, 11, 12, 14, 20, 8],    # Nichel
                [2, 0, 3, 5, 2, 1]])       # Impurità


from gurobipy import Model, GRB, quicksum

model = Model()

# Decision variables
x = [model.addVar(ub=Weights[i], obj=c)   for i, c in enumerate(Costs)]

#z = [model.addVar(vtype=GRB.BINARY, obj=Weights[i]*Costs[i]) for i in range(len(Costs))]

# Constraints
model.addConstr(quicksum(x[i] for i in range(len(x))) == 100)
#model.addConstr(quicksum(xi for xi in x) == 100)

model.addConstr(quicksum(Cs[0,i]/100*x[i] for i in range(len(x))) >= 65)
model.addConstr(quicksum(Cs[1,i]/100*x[i] for i in range(len(x))) == 18)
model.addConstr(quicksum(Cs[2,i]/100*x[i] for i in range(len(x))) == 10)
model.addConstr(quicksum(Cs[3,i]/100*x[i] for i in range(len(x))) <= 1)

#for i in range(len(x)):
#    model.addConstr(x[i] <= Weights[i]*z[i])

model.optimize()

print(f"Status: {model.Status}, ObjVal: {model.ObjVal}")
print('Solution vector', [xi.X for xi in x])
#print('Solution vector', [zi.X for zi in z])

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (mac64[arm] - Darwin 24.3.0 24D70)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 6 columns and 28 nonzeros
Model fingerprint: 0xa5935a04
Coefficient statistics:
  Matrix range     [1e-02, 1e+00]
  Objective range  [5e+01, 1e+02]
  Bounds range     [3e+01, 9e+01]
  RHS range        [1e+00, 1e+02]
Presolve time: 0.00s
Presolved: 5 rows, 6 columns, 28 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.6000000e+03   3.475000e+01   0.000000e+00      0s
       4    1.0567580e+04   0.000000e+00   0.000000e+00      0s

Solved in 4 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.056757991e+04
Status: 2, ObjVal: 10567.5799086758
Solution vector [0.0, 40.1826484018265, 0.0, 9.589041095890414, 1.8264840182648352, 48.40182648401826]


**EXERCISE 3:** First, write on paper an LP model to solve the steel production problem for Rossi's Steel company.

**EXERCISE 4:** Solve the LP using Gurobi.

### Exercise 2.6: The Magic Square Puzzle
The [Magic Square](https://en.wikipedia.org/wiki/Magic_square) puzzle asks you to place the digits from $1$ to $n^2$ into an $n \times n$ grid in such a way that the sum of the digits in each row, the sum of the digits in each column, and the sum of the digits on the two main diagonals are all equal to the same number.

You can play with a $4 \times 4$ puzzle on Google Sheets at [magic square link](https://docs.google.com/spreadsheets/d/1OcicQdKbZXpSV4ooXsbGC2OFR5cSgwgWgUimdwcT0qA/edit?usp=sharing).

**EXERCISE 5:** Write an ILP model to solve the magic square puzzle of size $n$.

In [ ]:
# Write your model here

## 2.7 Quiz: Gurobi Documentation Scavenger Hunt (Autograder)

This short quiz is designed to make you practice navigating the official Gurobi documentation.

How to use:
1. Fill in the code in each **Answer** cell (look for `TODO`).
2. Run **all** quiz cells from top to bottom.
3. The final cell prints a score summary.

Rules:
- Do not rename the required variables.
- Answers are short (usually a single word, constant name, or integer).

In [ ]:
# Grader helpers (do not edit)
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, List, Optional

@dataclass
class _Result:
    name: str
    ok: bool
    points: int
    message: str = ''

_RESULTS: List[_Result] = []
_SCORE = 0
_TOTAL = 0

def _grade(name: str, points: int, test_fn: Callable[[], None]) -> None:
    global _SCORE, _TOTAL
    _TOTAL += points
    try:
        test_fn()
    except AssertionError as e:
        _RESULTS.append(_Result(name=name, ok=False, points=points, message=str(e) or 'Assertion failed'))
    except Exception as e:
        _RESULTS.append(_Result(name=name, ok=False, points=points, message=f'{type(e).__name__}: {e}'))
    else:
        _SCORE += points
        _RESULTS.append(_Result(name=name, ok=True, points=points))

def _expect_raises(fn: Callable[[], None], exc_type: type[BaseException] = Exception) -> Optional[BaseException]:
    try:
        fn()
    except exc_type as e:
        return e
    except Exception as e:
        raise AssertionError(f'Expected {exc_type.__name__}, but got {type(e).__name__}: {e}') from e
    raise AssertionError(f'Expected {exc_type.__name__} to be raised, but nothing was raised')

def _norm_str(x: object) -> str:
    if x is None:
        return ''
    return ''.join(str(x).split()).lower()

### Q1 — `addVar`: objective coefficient keyword
Open the Gurobi docs for `Model.addVar` and find the keyword argument that sets the objective coefficient.

Docs: https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.addVar

Set `q1` to that keyword (as a string).

In [ ]:
# Answer (TODO)
q1 = None

In [ ]:
def _test_q1():
    assert _norm_str(q1) == 'obj', "q1 should be the string 'obj'"

_grade('Q1', 1, _test_q1)

### Q2 — `addVar`: default variable type
In the `Model.addVar` documentation, find the **default** variable type (`vtype`) when you do not specify it.

Docs: https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.addVar

Set `q2` to the default type (as a string), e.g., `GRB.CONTINUOUS`.

In [ ]:
# Answer (TODO)
q2 = None

In [ ]:
def _test_q2():
    s = _norm_str(q2)
    assert 'continuous' in s, "q2 should mention 'CONTINUOUS' (default vtype)"

_grade('Q2', 1, _test_q2)

### Q3 — `addVar`: default upper bound
In the `Model.addVar` documentation, find the default upper bound when you do not specify `ub`.

Docs: https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.addVar

Set `q3` to the default upper bound constant name (as a string), e.g., `GRB.INFINITY`.

In [ ]:
# Answer (TODO)
q3 = None

In [ ]:
def _test_q3():
    s = _norm_str(q3)
    assert 'infinity' in s, "q3 should mention 'INFINITY' (default ub)"

_grade('Q3', 1, _test_q3)

### Q4 — `addConstr`: constraint naming keyword
In the `Model.addConstr` documentation, find the keyword argument you can use to set a constraint's name.

Docs: https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html#Model.addConstr

Set `q4` to that keyword (as a string).

In [ ]:
# Answer (TODO)
q4 = None

In [ ]:
def _test_q4():
    assert _norm_str(q4) == 'name', "q4 should be the string 'name'"

_grade('Q4', 1, _test_q4)

### Q5 — Variable attribute for the solution value
After `model.optimize()`, each decision variable has an attribute that stores its primal solution value.

In the Gurobi `Var` documentation, find the attribute name used for this.

Docs: https://docs.gurobi.com/projects/optimizer/en/current/reference/python/var.html

Set `q5` to that attribute name (as a string), e.g., `X`.

In [ ]:
# Answer (TODO)
q5 = None

In [ ]:
def _test_q5():
    s = _norm_str(q5)
    assert s in ('x', '.x', 'var.x'), "q5 should be the attribute 'X' (e.g., 'X', '.X', or 'Var.X')"

_grade('Q5', 1, _test_q5)

### Q6 — Status code for an optimal solution
Gurobi stores an optimization result status in `model.Status`.
In the documentation, find the integer value corresponding to an optimal solution (i.e., `GRB.OPTIMAL`).

Docs: https://docs.gurobi.com/projects/optimizer/en/current/reference/python/grb.html

Set `q6` to that integer value.

In [ ]:
# Answer (TODO)
q6 = None

In [ ]:
def _test_q6():
    assert _norm_str(q6) == '2', 'q6 should be the integer value 2'

_grade('Q6', 1, _test_q6)

### Q7 — Time limit parameter name
Find the Gurobi parameter that sets a time limit (in seconds) for optimization.

Docs: https://docs.gurobi.com/projects/optimizer/en/current/reference/parameters.html

Set `q7` to the parameter name (as a string), e.g., `TimeLimit`.

In [ ]:
# Answer (TODO)
q7 = None

In [ ]:
def _test_q7():
    assert _norm_str(q7) == 'timelimit', "q7 should be the string 'TimeLimit'"

_grade('Q7', 1, _test_q7)

### Score summary
Run all quiz cells above first. This will show which questions passed and your total score.

In [ ]:
print('=== Score summary ===')
for r in _RESULTS:
    status = 'PASS' if r.ok else 'FAIL'
    earned = r.points if r.ok else 0
    msg = f" — {r.message}" if (not r.ok and r.message) else ''
    print(f"{r.name}: {status} (+{earned}/{r.points}){msg}")
print(f"TOTAL: {_SCORE}/{_TOTAL}")